In [ ]:
#LEARNING TO TOKENIZE (BASED ON KARPATHY'S MATERIALS)
#!/usr/bin/env python3

import requests
import torch
import re

# Source of information about tokanization:
# https://www.youtube.com/watch?v=QfNvhPx5Px8&t=
# https://docs.python.org/3/howto/unicode.html
# https://en.wikipedia.org/wiki/Byte-pair_encoding
# https://tiktokenizer.vercel.app/
# https://colab.research.google.com/drive/1y0KnCFZvGVf_odSfcNAws6kcDD7HsI0L?usp=sharing#scrollTo=pkAPaUCXOhvW
# https://sebastianraschka.com/blog/2025/bpe-from-scratch.html
# https://github.com/karpathy/minbpe

## 1. Baseline: Character-Level Tokenizer

The simplest tokenizer maps **each unique character** to an integer and back. The vocabulary
is just the set of distinct characters in the corpus. It is trivial to build but produces long
sequences and a tiny vocabulary — real models use **subword** tokenization (BPE), which we
build later in this notebook.

In [ ]:
# Dataset source:
data_set_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
#Download the dataset
dataset = requests.get(data_set_url, verify=False).text

#unique characters in the dataset
chars = sorted(list(set(dataset)))
vocab_size = len(chars)
print("All the unique characters:", "".join(chars))
print("Number of unique characters:", vocab_size)

# Create a mapping from characters to integers and vice versa
stoi = { ch:i for i,ch in enumerate(chars) }
itos = {i:ch for i,ch in enumerate(chars) }

# Encode and Decode functions
encode  = lambda s: [stoi[c] for c in s]
decode  = lambda l: "".join([itos[i] for i in l])

print("Encoded dataset:", encode("Hello World!3"))
print("Decoded dataset:", decode(encode("Hello World!3")))

c:\Program Files\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


All the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Number of unique characters: 65
Encoded dataset: [20, 43, 50, 50, 53, 1, 35, 53, 56, 50, 42, 2, 9]
Decoded dataset: Hello World!3


## 2. Representing Text: Unicode and Code Points

Before we can tokenize, we must turn characters into numbers. That is what **Unicode** does.

- **Unicode** assigns every character a unique **code point**, e.g. `A` → `U+0041`, `é` → `U+00E9`.
  Code points range from `0` to `0x10FFFF` (~1.1M possible values).
- A **code point is not bytes.** An **encoding** (an algorithm) converts code points to byte
  sequences and back. `A` (`U+0041`) is the single byte `\x41` in **UTF-8**, but `\x41\x00`
  in **UTF-16LE**.
- **Normalization caveat:** the same visible character can have multiple code-point sequences.
  `ê` can be one code point (`U+00EA`) or two (`e` + combining circumflex `U+0302`) — same
  output when printed, but different lengths. This matters when comparing strings.

In [11]:
#Encoding and decoding characters and texts:
print("Encoding and decoding characters and texts:")
s = 'café'
b_s = s.encode('utf-8')
print("Encoded string (UTF-8):", b_s,"Length:", len(b_s))
b_s = s.encode('utf-16')
print("Encoded string (UTF-16):", b_s,"Length:", len(b_s))

#Bytes rrepresention of café
cafe = bytes(s, 'utf-8')
print([cafe[i] for i in range(len(cafe))])


# Normalizing data
print("\nNormalizing data:")
from unicodedata import normalize, name,numeric
s1 = "café"
s2 = "cafe\u0301" # 'e' followed by a combining acute accent
print("s1:", s1, "s2:", s2)
print("s1 == s2:", s1 == s2)
print("s1 code points:", [name(c) for c in s1])
print("s2 code points:", [name(c) for c in s2])

s1_normalized = normalize('NFC', s1)
s2_normalized = normalize('NFC', s2)
print("s1_normalized:", s1_normalized, "s2_normalized:", s2_normalized)
print("s1_normalized == s2_normalized:", s1_normalized == s2_normalized)

print("é".casefold(),"e\u0301".casefold())

def nfc_equal(s1, s2):
    return normalize('NFC', s1) == normalize('NFC', s2)

def fold_equal(s1, s2):
    return normalize('NFC', s1).casefold() == normalize('NFC', s2).casefold()


# Unicode database
print("\nUnicode database:")
re_digit = re.compile(r'\d')
sample = '1\xbc\xb2\u0969\u136b\u216b\u2466\u2480\u3285'

print("Character\tDigit\tIsDigit\tIsNumeric\tNumeric Value\tName")
for c in sample:
    print("U+%04x" % ord(c), 
          c.center(6), 
          'red_dig' if re_digit.match(c) else '-', 
          'isdig' if c.isdigit() else '-',
          'isnum' if c.isnumeric() else '-',
          format(numeric(c),'5.2f'),
          name(c),sep='\t'
             
          )


Encoding and decoding characters and texts:
Encoded string (UTF-8): b'caf\xc3\xa9' Length: 5
Encoded string (UTF-16): b'\xff\xfec\x00a\x00f\x00\xe9\x00' Length: 10
[99, 97, 102, 195, 169]

Normalizing data:
s1: café s2: café
s1 == s2: False
s1 code points: ['LATIN SMALL LETTER C', 'LATIN SMALL LETTER A', 'LATIN SMALL LETTER F', 'LATIN SMALL LETTER E WITH ACUTE']
s2 code points: ['LATIN SMALL LETTER C', 'LATIN SMALL LETTER A', 'LATIN SMALL LETTER F', 'LATIN SMALL LETTER E', 'COMBINING ACUTE ACCENT']
s1_normalized: café s2_normalized: café
s1_normalized == s2_normalized: True
é é

Unicode database:
Character	Digit	IsDigit	IsNumeric	Numeric Value	Name
U+0031	  1   	red_dig	isdig	isnum	 1.00	DIGIT ONE
U+00bc	  ¼   	-	-	isnum	 0.25	VULGAR FRACTION ONE QUARTER
U+00b2	  ²   	-	isdig	isnum	 2.00	SUPERSCRIPT TWO
U+0969	  ३   	red_dig	isdig	isnum	 3.00	DEVANAGARI DIGIT THREE
U+136b	  ፫   	-	isdig	isnum	 3.00	ETHIOPIC DIGIT THREE
U+216b	  Ⅻ   	-	-	isnum	12.00	ROMAN NUMERAL TWELVE
U+2466	  ⑦   	

## 3. The Unicode Standard in Python

Python strings are sequences of Unicode code points. `chr(n)` maps a code point to its
character, `ord(c)` does the reverse. Below we explore code points (including the invisible
null character `\x00`) and the `\x` (byte, hex) vs `\u` (Unicode code point) escapes.

In [ ]:
#Return of chr(0): return of  string of length 1 of the Unicode code point 0, which is the null character.
#info: \x is used to represent a byte in hexadecimal format, represent from 00 to FF (0 to 255 in decimal). So, \x00 represents the null character, which is the character with Unicode code point 0.
#\u is used to represent a Unicode character in hexadecimal format, and it can represent characters with code points from 0000 to FFFF. So, \u0000 also represents the null character.
c = chr(0)  # This will create a string containing the null character.
print(c)  # This will not display anything because the null character is not visible when printed.
assert c == '\x00'  # The null character is represented as '\x00' in Python strings.
print(len('\x00'))  # This will print 1, as the string contains one character, the null character.
chr(0)

#Return of __repr__ and print: 
# The __repr__ method returns a printable python string that represents the object, and
# For a string containing the null character, the __repr__ will show it as '\x00', while print will not display anything for the null character.

c = chr(0)
r = c.__repr__() # or r = repr(c)
print(r)  # This will print '\x00', which is the representation of the null
print(len(r), list(r))
assert eval(r) == c  # This will be True, as eval will convert the string representation back to the original character.

In [ ]:
s = 'this is a test ' + chr(0) + ' string'
print(s)  # This will print 'this is a test  string', with the null character not visible.
print(s.__repr__())  # This will show the null character as '\x00' in the string representation.


#another example using the unicode point = 29275, which corresponds to the character 牛
c = chr(29275)  # This will create a string containing the character corresponding to Unicode code point 
unicode = '\u725B'  # This is the Unicode escape sequence for the character with code point 29275.
print(unicode)  # This will print 牛 , which is the character corresponding to Unicode code point 29275.
print(c)  # This will print the character corresponding to Unicode code point 29275, 
print(c.__repr__())  # This will show the character as '\u725B' in the string representation.
assert c == unicode  # This will be True, as both c and unicode represent the same character.
assert 29275 == ord(c)  # This will be True, as ord(c) returns the Unicode code point of the character in c, which is 29275.
assert 29275 == ord(unicode)  # This will be True, as ord(unicode) returns the Unicode code point of the character in unicode, which is also 29275.

s = 'this is a test ' + unicode + ' string'

## 4. Unicode Encodings: UTF-8 vs UTF-16 vs UTF-32

The same code point takes a different number of bytes depending on the encoding. **UTF-8** is
the de-facto standard: it is variable-length (1–4 bytes), ASCII-compatible, and compact for
common text. Modern tokenizers operate on **UTF-8 bytes**, which guarantees that *any* string
can be represented with a fixed 256-symbol byte alphabet (no "unknown character" problem).

In [ ]:
#Comparing UTF-8, UTF-16, and UTF-32 encodings of the character 牛 (Unicode code point 29275):
#UTF-8 encoding of 牛: 0xE7 0x89 0x9B
unicode = '\u725B'
utf8_encoded = unicode.encode('utf-8')
print(len(utf8_encoded))  # This will print the length of the UTF-8 encoded bytes, which is 3 bytes for the character 牛.
print(utf8_encoded)  # This will print the UTF-8 encoded bytes for the character 牛, which is b'\xe7\x89\x9b'.
#UTF-16 encoding of 牛: 0x72 0x5B
utf16_encoded = unicode.encode('utf-16')
print(len(utf16_encoded))  # This will print the length of the UTF-16 encoded bytes, which is 4 bytes (including the Byte Order Mark).  
print(utf16_encoded)  # This will print the UTF-16 encoded bytes for the character 牛, which is b'\xff\xfeR[' (with a Byte Order Mark).

#
s = "hello! こんにちは!"
print(s.encode('utf-8'))  # This will print the UTF-8 encoded bytes for the string, which includes both ASCII and Japanese characters.
print(s.encode('utf-16'))  # This will print the UTF-16 encoded bytes
b = s.encode('utf-8')
#to decode long string of bytes, we can use the following code:
#  we can not convert the bytes directly to a string using decode, because it will not handle the multi-byte characters correctly. 
# Instead, we can iterate through the bytes and convert each byte to a character, then join them together to form the original string.
b''.join(bytes([b[i]]) for i in range(len(b))).decode('utf-8')

In [ ]:
text ="""" Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception. A few months ago, I got interested in Unicode and decided to spend some time learning more about it in detail. In this article, I’ll give an introduction to it from a programmer’s point of view. I’m going to focus on the character set and what’s involved in working with strings and files of Unicode text. However, in this article I’m not going to talk about fonts, text layout/shaping/rendering, or localization in detail—those are separate issues, beyond my scope (and knowledge) here. Diversity and Inherent Complexity The Unicode Codespace Codespace Allocation Scripts Usage Frequency Encodings UTF-8 UTF-16 Combining Marks Canonical Equivalence Normalization Forms Grapheme Clusters And More… Diversity and Inherent Complexity As soon as you start to study Unicode, it becomes clear that it represents a large jump in complexity over character sets like ASCII that you may be more familiar with. It’s not just that Unicode contains a much larger number of characters, although that’s part of it. Unicode also has a great deal of internal structure, features, and special cases, making it much more than what one might expect a mere “character set” to be. We’ll see some of that later in this article. When confronting all this complexity, especially as an engineer, it’s hard not to find oneself asking, “Why do we need all this? Is this really necessary? Couldn’t it be simplified?” However, Unicode aims to faithfully represent the entire world’s writing systems. The Unicode Consortium’s stated goal is “enabling people around the world to use computers in any language”. And as you might imagine, the diversity of written languages is immense! To date, Unicode supports 135 different scripts, covering some 1100 languages, and there’s still a long tail of over 100 unsupported scripts, both modern and historical, which people are still working to add. Given this enormous diversity, it’s inevitable that representing it is a complicated project. Unicode embraces that diversity, and accepts the complexity inherent in its mission to include all human writing systems. It doesn’t make a lot of trade-offs in the name of simplification, and it makes exceptions to its own rules where necessary to further its mission. Moreover, Unicode is committed not just to supporting texts in any single language, but also to letting multiple languages coexist within one text—which introduces even more complexity. Most programming languages have libraries available to handle the gory low-level details of text manipulation, but as a programmer, you’ll still need to know about certain Unicode features in order to know when and how to apply them. It may take some time to wrap your head around it all, but don’t be discouraged—think about the billions of people for whom your software will be more accessible through supporting text in their language. Embrace the complexity! The Unicode Codespace Let’s start with some general orientation. The basic elements of Unicode—its “characters”, although that term isn’t quite right—are called code points. Code points are identified by number, customarily written in hexadecimal with the prefix “U+”, such as U+0041 “A” latin capital letter a or U+03B8 “θ” greek small letter theta. Each code point also has a short name, and quite a few other properties, specified in the Unicode Character Database. The set of all possible code points is called the codespace. The Unicode codespace consists of 1,114,112 code points. However, only 128,237 of them—about 12% of the codespace—are actually assigned, to date. There’s plenty of room for growth! Unicode also reserves an additional 137,468 code points as “private use” areas, which have no standardized meaning and are available for individual applications to define for their own purposes. Codespace Allocation To get a feel for how the codespace is laid out, it’s helpful to visualize it. Below is a map of the entire codespace, with one pixel per code point. It’s arranged in tiles for visual coherence; each small square is 16×16 = 256 code points, and each large square is a “plane” of 65,536 code points. There are 17 planes altogether. Map of the Unicode codespace (click to zoom) White represents unassigned space. Blue is assigned code points, green is private-use areas, and the small red area is surrogates (more about those later). As you can see, the assigned code points are distributed somewhat sparsely, but concentrated in the first three planes. Plane 0 is also known as the “Basic Multilingual Plane”, or BMP. The BMP contains essentially all the characters needed for modern text in any script, including Latin, Cyrillic, Greek, Han (Chinese), Japanese, Korean, Arabic, Hebrew, Devanagari (Indian), and many more. (In the past, the codespace was just the BMP and no more—Unicode was originally conceived as a straightforward 16-bit encoding, with only 65,536 code points. It was expanded to its current size in 1996. However, the vast majority of code points in modern text belong to the BMP.) Plane 1 contains historical scripts, such as Sumerian cuneiform and Egyptian hieroglyphs, as well as emoji and various other symbols. Plane 2 contains a large block of less-common and historical Han characters. The remaining planes are empty, except for a small number of rarely-used formatting characters in Plane 14; planes 15–16 are reserved entirely for private use. Scripts Let’s zoom in on the first three planes, since that’s where the action is: Map of scripts in Unicode planes 0–2 (click to zoom) This map color-codes the 135 different scripts in Unicode. You can see how Han () and Korean () take up most of the range of the BMP (the left large square). By contrast, all of the European, Middle Eastern, and South Asian scripts fit into the first row of the BMP in this diagram. Many areas of the codespace are adapted or copied from earlier encodings. For example, the first 128 code points of Unicode are just a copy of ASCII. This has clear benefits for compatibility—it’s easy to losslessly convert texts from smaller encodings into Unicode (and the other direction too, as long as no characters outside the smaller encoding are used). Usage Frequency One more interesting way to visualize the codespace is to look at the distribution of usage—in other words, how often each code point is actually used in real-world texts. Below is a heat map of planes 0–2 based on a large sample of text from Wikipedia and Twitter (all languages). Frequency increases from black (never seen) through red and yellow to white. Heat map of code point usage frequency in Unicode planes 0–2 (click to zoom) You can see that the vast majority of this text sample lies in the BMP, with only scattered usage of code points from planes 1–2. The biggest exception is emoji, which show up here as the several bright squares in the bottom row of plane 1. Encodings We’ve seen that Unicode code points are abstractly identified by their index in the codespace, ranging from U+0000 to U+10FFFF. But how do code points get represented as bytes, in memory or in a file? The most convenient, computer-friendliest (and programmer-friendliest) thing to do would be to just store the code point index as a 32-bit integer. This works, but it consumes 4 bytes per code point, which is sort of a lot. Using 32-bit ints for Unicode will cost you a bunch of extra storage, memory, and performance in bandwidth-bound scenarios, if you work with a lot of text. Consequently, there are several more-compact encodings for Unicode. The 32-bit integer encoding is officially called UTF-32 (UTF = “Unicode Transformation Format”), but it’s rarely used for storage. At most, it comes up sometimes as a temporary internal representation, for examining or operating on the code points in a string. Much more commonly, you’ll see Unicode text encoded as either UTF-8 or UTF-16. These are both variable-length encodings, made up of 8-bit or 16-bit units, respectively. In these schemes, code points with smaller index values take up fewer bytes, which saves a lot of memory for typical texts. The trade-off is that processing UTF-8/16 texts is more programmatically involved, and likely slower. UTF-8 In UTF-8, each code point is stored using 1 to 4 bytes, based on its index value. UTF-8 uses a system of binary prefixes, in which the high bits of each byte mark whether it’s a single byte, the beginning of a multi-byte sequence, or a continuation byte; the remaining bits, concatenated, give the code point index. This table shows how it works: UTF-8 (binary)	Code point (binary)	Range 0xxxxxxx	xxxxxxx	U+0000–U+007F 110xxxxx 10yyyyyy	xxxxxyyyyyy	U+0080–U+07FF 1110xxxx 10yyyyyy 10zzzzzz	xxxxyyyyyyzzzzzz	U+0800–U+FFFF 11110xxx 10yyyyyy 10zzzzzz 10wwwwww	xxxyyyyyyzzzzzzwwwwww	U+10000–U+10FFFF A handy property of UTF-8 is that code points below 128 (ASCII characters) are encoded as single bytes, and all non-ASCII code points are encoded using sequences of bytes 128–255. This has a couple of nice consequences. First, any strings or files out there that are already in ASCII can also be interpreted as UTF-8 without any conversion. Second, lots of widely-used string programming idioms—such as null termination, or delimiters (newlines, tabs, commas, slashes, etc.)—will just work on UTF-8 strings. ASCII bytes never occur inside the encoding of non-ASCII code points, so searching byte-wise for a null terminator or a delimiter will do the right thing. Thanks to this convenience, it’s relatively simple to extend legacy ASCII programs and APIs to handle UTF-8 strings. UTF-8 is very widely used in the Unix/Linux and Web worlds, and many programmers argue UTF-8 should be the default encoding everywhere. However, UTF-8 isn’t a drop-in replacement for ASCII strings in all respects. For instance, code that iterates over the “characters” in a string will need to decode UTF-8 and iterate over code points (or maybe grapheme clusters—more about those later), not bytes. When you measure the “length” of a string, you’ll need to think about whether you want the length in bytes, the length in code points, the width of the text when rendered, or something else. UTF-16 The other encoding that you’re likely to encounter is UTF-16. It uses 16-bit words, with each code point stored as either 1 or 2 words. Like UTF-8, we can express the UTF-16 encoding rules in the form of binary prefixes: UTF-16 (binary)	Code point (binary)	Range xxxxxxxxxxxxxxxx	xxxxxxxxxxxxxxxx	U+0000–U+FFFF 110110xxxxxxxxxx 110111yyyyyyyyyy	xxxxxxxxxxyyyyyyyyyy + 0x10000	U+10000–U+10FFFF A more common way that people talk about UTF-16 encoding, though, is in terms of code points called “surrogates”. All the code points in the range U+D800–U+DFFF—or in other words, the code points that match the binary prefixes 110110 and 110111 in the table above—are reserved specifically for UTF-16 encoding, and don’t represent any valid characters on their own. They’re only meant to occur in the 2-word encoding pattern above, which is called a “surrogate pair”. Surrogate code points are illegal in any other context! They’re not allowed in UTF-8 or UTF-32 at all. Historically, UTF-16 is a descendant of the original, pre-1996 versions of Unicode, in which there were only 65,536 code points. The original intention was that there would be no different “encodings”; Unicode was supposed to be a straightforward 16-bit character set. Later, the codespace was expanded to make room for a long tail of less-common (but still important) Han characters, which the Unicode designers didn’t originally plan for. Surrogates were then introduced, as—to put it bluntly—a kludge, allowing 16-bit encodings to access the new code points. Today, Javascript uses UTF-16 as its standard string representation: if you ask for the length of a string, or iterate over it, etc., the result will be in UTF-16 words, with any code points outside the BMP expressed as surrogate pairs. UTF-16 is also used by the Microsoft Win32 APIs; though Win32 supports either 8-bit or 16-bit strings, the 8-bit version unaccountably still doesn’t support UTF-8—only legacy code-page encodings, like ANSI. This leaves UTF-16 as the only way to get proper Unicode support in Windows. (Update: in Win10 version 1903, they finally added UTF-8 support to the 8-bit APIs! 😊) By the way, UTF-16’s words can be stored either little-endian or big-endian. Unicode has no opinion on that issue, though it does encourage the convention of putting U+FEFF zero width no-break space at the top of a UTF-16 file as a byte-order mark, to disambiguate the endianness. (If the file doesn’t match the system’s endianness, the BOM will be decoded as U+FFFE, which isn’t a valid code point.) Combining Marks In the story so far, we’ve been focusing on code points. But in Unicode, a “character” can be more complicated than just an individual code point! Unicode includes a system for dynamically composing characters, by combining multiple code points together. This is used in various ways to gain flexibility without causing a huge combinatorial explosion in the number of code points. In European languages, for example, this shows up in the application of diacritics to letters. Unicode supports a wide range of diacritics, including acute and grave accents, umlauts, cedillas, and many more. All these diacritics can be applied to any letter of any alphabet—and in fact, multiple diacritics can be used on a single letter. If Unicode tried to assign a distinct code point to every possible combination of letter and diacritics, things would rapidly get out of hand. Instead, the dynamic composition system enables you to construct the character you want, by starting with a base code point (the letter) and appending additional code points, called “combining marks”, to specify the diacritics. When a text renderer sees a sequence like this in a string, it automatically stacks the diacritics over or under the base letter to create a composed character. For example, the accented character “Á” can be expressed as a string of two code points: U+0041 “A” latin capital letter a plus U+0301 “◌́” combining acute accent. This string automatically gets rendered as a single character: “Á”. Now, Unicode does also include many “precomposed” code points, each representing a letter with some combination of diacritics already applied, such as U+00C1 “Á” latin capital letter a with acute or U+1EC7 “ệ” latin small letter e with circumflex and dot below. I suspect these are mostly inherited from older encodings that were assimilated into Unicode, and kept around for compatibility. In practice, there are precomposed code points for most of the common letter-with-diacritic combinations in European-script languages, so they don’t use dynamic composition that much in typical text. Still, the system of combining marks does allow for an arbitrary number of diacritics to be stacked on any base character. The reductio-ad-absurdum of this is Zalgo text, which works by ͖͟ͅr͞aṋ̫̠̖͈̗d͖̻̹óm̪͙͕̗̝ļ͇̰͓̳̫ý͓̥̟͍ ̕s̫t̫̱͕̗̰̼̘͜a̼̩͖͇̠͈̣͝c̙͍k̖̱̹͍͘i̢n̨̺̝͇͇̟͙ģ̫̮͎̻̟ͅ ̕n̼̺͈͞u̮͙m̺̭̟̗͞e̞͓̰̤͓̫r̵o̖ṷs҉̪͍̭̬̝̤ ̮͉̝̞̗̟͠d̴̟̜̱͕͚i͇̫̼̯̭̜͡ḁ͙̻̼c̲̲̹r̨̠̹̣̰̦i̱t̤̻̤͍͙̘̕i̵̜̭̤̱͎c̵s ͘o̱̲͈̙͖͇̲͢n͘ ̜͈e̬̲̠̩ac͕̺̠͉h̷̪ ̺̣͖̱ḻ̫̬̝̹ḙ̙̺͙̭͓̲t̞̞͇̲͉͍t̷͔̪͉̲̻̠͙e̦̻͈͉͇r͇̭̭̬͖,̖́ ̜͙͓̣̭s̘̘͈o̱̰̤̲ͅ ̛̬̜̙t̼̦͕̱̹͕̥h̳̲͈͝ͅa̦t̻̲ ̻̟̭̦̖t̛̰̩h̠͕̳̝̫͕e͈̤̘͖̞͘y҉̝͙ ̷͉͔̰̠o̞̰v͈͈̳̘͜er̶f̰͈͔ḻ͕̘̫̺̲o̲̭͙͠ͅw̱̳̺ ͜t̸h͇̭͕̳͍e̖̯̟̠ ͍̞̜͔̩̪͜ļ͎̪̲͚i̝̲̹̙̩̹n̨̦̩̖ḙ̼̲̼͢ͅ ̬͝s̼͚̘̞͝p͙̘̻a̙c҉͉̜̤͈̯̖i̥͡n̦̠̱͟g̸̗̻̦̭̮̟ͅ ̳̪̠͖̳̯̕a̫͜n͝d͡ ̣̦̙ͅc̪̗r̴͙̮̦̹̳e͇͚̞͔̹̫͟a̙̺̙ț͔͎̘̹ͅe̥̩͍ a͖̪̜̮͙̹n̢͉̝ ͇͉͓̦̼́a̳͖̪̤̱p̖͔͔̟͇͎͠p̱͍̺ę̲͎͈̰̲̤̫a̯͜r̨̮̫̣̘a̩̯͖n̹̦̰͎̣̞̞c̨̦̱͔͎͍͖e̬͓͘ ̤̰̩͙̤̬͙o̵̼̻̬̻͇̮̪f̴ ̡̙̭͓͖̪̤“̸͙̠̼c̳̗͜o͏̼͙͔̮r̞̫̺̞̥̬ru̺̻̯͉̭̻̯p̰̥͓̣̫̙̤͢t̳͍̳̖ͅi̶͈̝͙̼̙̹o̡͔n̙̺̹̖̩͝ͅ”̨̗͖͚̩.̯͓ A few other places where dynamic character composition shows up in Unicode: Vowel-pointing notation in Arabic and Hebrew. In these languages, words are normally spelled with some of their vowels left out. They then have diacritic notation to indicate the vowels (used in dictionaries, language-teaching materials, children’s books, and such). These diacritics are expressed with combining marks. A Hebrew example, with niqqud:	אֶת דַלְתִּי הֵזִיז הֵנִיעַ, קֶטֶב לִשְׁכַּתִּי יָשׁוֹד Normal writing (no niqqud):	את דלתי הזיז הניע, קטב לשכתי ישוד Devanagari, the script used to write Hindi, Sanskrit, and many other South Asian languages, expresses certain vowels as combining marks attached to consonant letters. For example, “ह” + “​ि” = “हि” (“h” + “i” = “hi”). Korean characters stand for syllables, but they are composed of letters called jamo that stand for the vowels and consonants in the syllable. While there are code points for precomposed Korean syllables, it’s also possible to dynamically compose them by concatenating their jamo. For example, “ᄒ” + “ᅡ” + “ᆫ” = “한” (“h” + “a” + “n” = “han”). Canonical Equivalence In Unicode, precomposed characters exist alongside the dynamic composition system. A consequence of this is that there are multiple ways to express “the same” string—different sequences of code points that result in the same user-perceived characters. For example, as we saw earlier, we can express the character “Á” either as the single code point U+00C1, or as the string of two code points U+0041 U+0301. Another source of ambiguity is the ordering of multiple diacritics in a single character. Diacritic order matters visually when two diacritics apply to the same side of the base character, e.g. both above: “ǡ” (dot, then macron) is different from “ā̇” (macron, then dot). However, when diacritics apply to different sides of the character, e.g. one above and one below, then the order doesn’t affect rendering. Moreover, a character with multiple diacritics might have one of the diacritics precomposed and others expressed as combining marks. For example, the Vietnamese letter “ệ” can be expressed in five different ways: Fully precomposed: U+1EC7 “ệ” Partially precomposed: U+1EB9 “ẹ” + U+0302 “◌̂” Partially precomposed: U+00EA “ê” + U+0323 “◌̣” Fully decomposed: U+0065 “e” + U+0323 “◌̣” + U+0302 “◌̂” Fully decomposed: U+0065 “e” + U+0302 “◌̂” + U+0323 “◌̣” Unicode refers to set of strings like this as “canonically equivalent”. Canonically equivalent strings are supposed to be treated as identical for purposes of searching, sorting, rendering, text selection, and so on. This has implications for how you implement operations on text. For example, if an app has a “find in file” operation and the user searches for “ệ”, it should, by default, find occurrences of any of the five versions of “ệ” above! Normalization Forms To address the problem of “how to handle canonically equivalent strings”, Unicode defines several normalization forms: ways of converting strings into a canonical form so that they can be compared code-point-by-code-point (or byte-by-byte). The “NFD” normalization form fully decomposes every character down to its component base and combining marks, taking apart any precomposed code points in the string. It also sorts the combining marks in each character according to their rendered position, so e.g. diacritics that go below the character come before the ones that go above the character. (It doesn’t reorder diacritics in the same rendered position, since their order matters visually, as previously mentioned.) The “NFC” form, conversely, puts things back together into precomposed code points as much as possible. If an unusual combination of diacritics is called for, there may not be any precomposed code point for it, in which case NFC still precomposes what it can and leaves any remaining combining marks in place (again ordered by rendered position, as in NFD). There are also forms called NFKD and NFKC. The “K” here refers to compatibility decompositions, which cover characters that are “similar” in some sense but not visually identical. However, I’m not going to cover that here. Grapheme Clusters As we’ve seen, Unicode contains various cases where a thing that a user thinks of as a single “character” might actually be made up of multiple code points under the hood. Unicode formalizes this using the notion of a grapheme cluster: a string of one or more code points that constitute a single “user-perceived character”. UAX #29 defines the rules for what, precisely, qualifies as a grapheme cluster. It’s approximately “a base code point followed by any number of combining marks”, but the actual definition is a bit more complicated; it accounts for things like Korean jamo, and emoji ZWJ sequences. The main thing grapheme clusters are used for is text editing: they’re often the most sensible unit for cursor placement and text selection boundaries. Using grapheme clusters for these purposes ensures that you can’t accidentally chop off some diacritics when you copy-and-paste text, that left/right arrow keys always move the cursor by one visible character, and so on. Another place where grapheme clusters are useful is in enforcing a string length limit—say, on a database field. While the true, underlying limit might be something like the byte length of the string in UTF-8, you wouldn’t want to enforce that by just truncating bytes. At a minimum, you’d want to “round down” to the nearest code point boundary; but even better, round down to the nearest grapheme cluster boundary. Otherwise, you might be corrupting the last character by cutting off a diacritic, or interrupting a jamo sequence or ZWJ sequence. And More… There’s much more that could be said about Unicode from a programmer’s perspective! I haven’t gotten into such fun topics as case mapping, collation, compatibility decompositions and confusables, Unicode-aware regexes, or bidirectional text. Nor have I said anything yet about implementation issues—how to efficiently store and look-up data about the sparsely-assigned code points, or how to optimize UTF-8 decoding, string comparison, or NFC normalization. Perhaps I’ll return to some of those things in future posts. Unicode is a fascinating and complex system. It has a many-to-one mapping between bytes and code points, and on top of that a many-to-one (or, under some circumstances, many-to-many) mapping between code points and “characters”. It has oddball special cases in every corner. But no one ever claimed that representing all written languages was going to be easy, and it’s clear that we’re never going back to the bad old days of a patchwork of incompatible encodings. Further reading: The Unicode Standard UTF-8 Everywhere Manifesto Dark corners of Unicode by Eevee ICU (International Components for Unicode)—C/C++/Java libraries implementing many Unicode algorithms and related things Python 3 Unicode Howto Google Noto Fonts—set of fonts intended to cover all assigned code points"""

## 5. Byte-Pair Encoding (BPE)

Character tokens give short vocabularies but very long sequences; word tokens give short
sequences but huge, brittle vocabularies (and fail on unseen words). **BPE** is the middle
ground used by GPT.

**The algorithm** (a simple compression scheme applied to UTF-8 bytes):

1. Start from the 256 raw bytes as the base vocabulary.
2. Count all adjacent token pairs; find the **most frequent** pair.
3. **Merge** it into a new token with a fresh id, and record the merge.
4. Repeat for a fixed number of merges (the target vocabulary size).

Frequent sequences (common words, suffixes) become single tokens, while rare strings fall
back to smaller pieces — so the tokenizer **never fails** on unseen input. Below we implement
both directions: **encode** (apply merges) and **decode** (invert them).

In [ ]:
#BPE PAIR ENCODING (BPE)
#BPE is a simple data compression technique that iteratively replaces the most frequent pair of bytes in a sequence with a single, unused byte. 
# In the context of tokenization for language models, BPE is used to create a more efficient representation of text by merging frequently co-occurring characters or subwords into single tokens. 
# This allows the model to handle a larger vocabulary while keeping the number of tokens manageable.

s = text
vocab_size = 276
num_merge = vocab_size-256
tokens = s.encode('utf-8') #raw bytes of text
tokens = list(map(int, tokens)) #convert bytes to integers

from collections import Counter

def merge_pair(tokens, pair, new_token):
    a, b = pair
    merged = []
    i = 0
    n = len(tokens)
    while i < n:
        if i + 1 < n and tokens[i] == a and tokens[i + 1] == b:
            merged.append(new_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged

def bpe_pair_encoding(tokens, num_merge, *, start_id=None, verbose=False, min_frequency=2):
    """Learn BPE merges by repeatedly merging the most frequent adjacent pair.
    Args:
        tokens: Sequence[int] token ids (bytes 0..255 or already-merged ids).
        num_merge: Max number of merges to learn.
        start_id: First id to assign to merged tokens. Defaults to max(256, max(tokens)+1).
        verbose: If True, prints progress.
        min_frequency: Stop early if the best pair occurs fewer times than this.

    Returns:
        (merged, merges) where merges is a list of ((a,b), new_id) in application order.
    """
    merged = list(tokens) #copy of tokens that we will modify in-place
    if num_merge <= 0 or len(merged) < 2:
        return merged, []

    next_id = start_id
    if next_id is None:
        next_id = max(256, max(merged) + 1)

    merges = {}
    for merge_index in range(num_merge):
        # Counts adjacent pairs: (token_i, token_{i+1}) -> frequency
        counts = Counter(zip(merged, merged[1:]))
        if not counts:
            break

        best_pair, best_freq = max(
            counts.items(),
            # Deterministic tie-break: pick the lexicographically smallest pair among the best.
            key=lambda kv: (kv[1], -kv[0][0], -kv[0][1]), #kv = ((a, b), freq) -> (freq, -a, -b)
        )
        if best_freq < min_frequency:
            break

        if verbose:
            print(f"[{merge_index+1}/{num_merge}] merging {best_pair} (freq={best_freq}) -> {next_id}")

        merged = merge_pair(merged, best_pair, next_id)
        merges[best_pair] = next_id
        next_id += 1

        if len(merged) < 2:
            break

    return merged, merges


ids, merges = bpe_pair_encoding(tokens, num_merge, verbose=True)
print("tokens length:", len(tokens))
print("ids length", len(ids))
print(f"Compression ratio: {len(tokens)/ len(ids):.2f}X")

[1/20] merging (101, 32) (freq=645) -> 256
[2/20] merging (105, 110) (freq=445) -> 257
[3/20] merging (115, 32) (freq=422) -> 258
[4/20] merging (116, 104) (freq=337) -> 259
[5/20] merging (101, 114) (freq=293) -> 260
[6/20] merging (99, 111) (freq=289) -> 261
[7/20] merging (116, 32) (freq=285) -> 262
[8/20] merging (226, 128) (freq=253) -> 263
[9/20] merging (44, 32) (freq=242) -> 264
[10/20] merging (97, 110) (freq=230) -> 265
[11/20] merging (111, 114) (freq=214) -> 266
[12/20] merging (100, 32) (freq=213) -> 267
[13/20] merging (97, 114) (freq=180) -> 268
[14/20] merging (101, 110) (freq=173) -> 269
[15/20] merging (257, 103) (freq=165) -> 270
[16/20] merging (261, 100) (freq=164) -> 271
[17/20] merging (46, 32) (freq=154) -> 272
[18/20] merging (121, 32) (freq=154) -> 273
[19/20] merging (97, 108) (freq=146) -> 274
[20/20] merging (259, 256) (freq=144) -> 275
tokens length: 24469
ids length 19321
Compression ratio: 1.27X


In [ ]:
# Decoding BPE tokens back to text

vocab = {i: bytes([i]) for i in range(256)} #initial vocab maps byte values to single-byte bytes objects
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b] #merged token maps to concatenation of its components

def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

print(decode([97, 98, 170])) #ab


ab�


In [ ]:
# Encoding text to BPE tokens
from collections import Counter

def encode(text, merges):
    #given a text string, return the list of integers(tokens)
    tokens = list(text.encode("utf-8"))
    while len(tokens) > 2:
        stats = Counter(zip(tokens, tokens[1:])) #make pairs with frequency counts
        #find the pair with the smallest merge id (most recently merged )
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge_pair(tokens, pair, idx)
    return tokens

print(encode("hello world!", merges))
#print(encode("!$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789", merges))
print(decode(encode("hello world!", merges)))

Initial tokens: [104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33]
Tokens after merging: [104, 101, 108, 108, 111, 32, 119, 266, 108, 100, 33]
[104, 101, 108, 108, 111, 32, 119, 266, 108, 100, 33]
Initial tokens: [104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33]
Tokens after merging: [104, 101, 108, 108, 111, 32, 119, 266, 108, 100, 33]
hello world!


##### Forced Split Using Regex Patterns (GPT Series)
Source: https://github.com/openai/gpt-2/blob/master/src/encoder.py (line 53)

In [25]:
import regex as re
#Pattern from gpt2's tokenizer, which is a byte-level BPE tokenizer. It splits text into tokens based on the following rules:
#1. It recognizes contractions and common suffixes like 's, 't, 're, 've, 'm, 'll, and 'd as separate tokens.
#2. It matches sequences of letters (Unicode letter characters) as tokens, allowing for an optional leading space.| ?\p{L}+|
#3. It matches sequences of digits (Unicode number characters) as tokens, also allowing for an optional leading space.| ?\p{N}+|
#4. It matches sequences of non-whitespace, non-letter, and non-digit characters as tokens, again allowing for an optional leading space.| ?[^\s\p{L}\p{N}]+|
#5. It matches sequences of whitespace characters as tokens, but only if they are not followed by a non-whitespace character (i.e., it captures trailing whitespace as a token).|\s+(?!\S)|\s+
gpt2pat = pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello've world123 how's are     you !!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', '    ', ' you', ' !!?']


## 6. Putting It Together: The Tokenizer Classes

Everything above — UTF-8 bytes, BPE merges, `encode`/`decode`, the GPT-2 regex split — is
bundled into a small, reusable class hierarchy in the repository's `tokenizer/` package.
These classes are the "productionized" version of the from-scratch functions we wrote above.

### Class hierarchy

```
Tokenizer (ABC, base.py)          abstract: defines the interface + shared utilities
├── BasicTokenizer (basic.py)     plain BPE over raw UTF-8 bytes (no pre-split)
├── RegexTokenizer (regex.py)     BPE + GPT-2/4 regex pre-split, special tokens, parallel training
│     └── GPT4Tokenizer (gpt4.py) loads GPT-4's pretrained merges (tiktoken cl100k)
├── CharTokenizer / ByteTokenizer (char.py)   trivial char-/byte-level baselines
└── load_gpt2_tokenizer() (gpt2.py)           helper that loads GPT-2's pretrained vocab + merges
```

> Note: `BasicTokenizer` and `RegexTokenizer` both inherit **directly** from `Tokenizer`;
> only `GPT4Tokenizer` extends `RegexTokenizer`.

### The abstract base — `Tokenizer`

It holds the shared state and utilities so each subclass only has to implement the algorithm:

- **State:** `merges` `(a, b) → new_id`, `vocab` `id → bytes`, `pattern`, `special_tokens`.
- **Must implement** (`@abstractmethod`): `train`, `train_iterator`, `encode`, `decode`.
- **Provided for free:** `build_vocab()` (rebuilds `id → bytes` from the merges),
  `save`/`load` (persist a trained tokenizer), `get_ratio()` (compression = bytes ÷ tokens),
  and `visualise_tokens()` (pretty-print the pieces).

### The subclasses

| Class | Pre-split | Trainable | Use case |
|-------|-----------|-----------|----------|
| `BasicTokenizer` | none | yes | learn BPE on raw bytes — closest to the code above |
| `RegexTokenizer` | GPT-2/4 regex | yes | the practical one: words/digits/punctuation never merge across boundaries |
| `GPT4Tokenizer` | GPT-4 regex | no (loads cl100k) | reuse GPT-4's exact vocabulary |
| `load_gpt2_tokenizer()` | GPT-2 regex | no (loads pretrained) | reuse GPT-2's exact vocabulary |
| `CharTokenizer` / `ByteTokenizer` | — | trivial | char/byte baselines (see Section 1) |

> The model in this repo is trained on tokens produced by these classes — see `train/train.py`.

Below we train a `RegexTokenizer` on a Portuguese corpus and round-trip a sentence.

In [2]:
# RegexTokenizer = the from-scratch BPE above, packaged with a GPT-2-style regex pre-split.
# Inheritance: Tokenizer (ABC) -> RegexTokenizer
import sys, os
sys.path.append(os.path.abspath('..'))
from data import ReadTextFile
from tokenizer import RegexTokenizer

# 1. Load a corpus (one string per line)
ds = ReadTextFile('../database/wikipedia_pt_1M.txt')
corpus = ds.get_text_lines()

# 2. Train BPE: 256 base bytes + (vocab_size - 256) learned merges
tokenizer = RegexTokenizer()
tokenizer.train(corpus, vocab_size=300)
print(f"Vocabulary size: {len(tokenizer.vocab)}  ({len(tokenizer.merges)} learned merges)")

# 3. Encode / decode round-trip
text = "Olá, mundo! Este é um teste de tokenização."
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(f"\nText      : {text}")
print(f"Token ids : {ids}")
print(f"Decoded   : {decoded}")
print(f"Round-trip OK? {decoded == text}")
print(f"Compression: {len(text.encode('utf-8'))} bytes -> {len(ids)} tokens "
      f"({tokenizer.get_ratio(text, ids):.2f} bytes/token)")

# 4. Visualize the actual pieces the text was split into
print("\nToken pieces:")
tokenizer.visualise_tokens(ids)

Vocabulary size: 300  (44 learned merges)

Text      : Olá, mundo! Este é um teste de tokenização.
Token ids : [79, 108, 195, 161, 44, 32, 109, 117, 296, 111, 33, 32, 69, 298, 101, 32, 285, 299, 109, 287, 265, 116, 101, 259, 287, 111, 107, 101, 110, 105, 122, 97, 283, 278, 46]
Decoded   : Olá, mundo! Este é um teste de tokenização.
Round-trip OK? True
Compression: 47 bytes -> 35 tokens (1.34 bytes/token)

Token pieces:
Ol��, mundo! Este é um teste de tokenização.
